# Vivado API Example

This notebook uploads a VHDL project to the Vivado API server, launches synthesis and simulation, and prints the outputs.

In [ ]:
BASE_URL = "http://ricpc:8000"
API_KEY = "43567543243653856"

headers = { "Authorization": f"Bearer {API_KEY}" }

## Check API status

In [4]:
import requests
import json

r = requests.get(f"{BASE_URL}/health")

print("Status code:", r.status_code)
print("\nJSON response:")
print(json.dumps(r.json(), indent=4))

Status code: 200

JSON response:
{
    "status": "ok",
    "vivado_path": "C:/Xilinx/2025.1/Vivado/bin/vivado.bat",
    "vivado_available": true,
    "workspace_dir": "C:\\vivado_api",
    "auth_enabled": true
}


## Upload all VHDL sources

In [7]:
from pathlib import Path
import requests

src_files = sorted(Path("src").glob("*.vhd"))
tb_files = sorted(Path("tb").glob("*.vhd"))
xdc_files = sorted(Path("constraints").glob("*.xdc"))

all_files = list(src_files) + list(tb_files) + list(xdc_files)

print("Files to upload:")
for f in all_files:
    print("-", f)

multipart = []

for f in all_files:
    multipart.append(
        ("files", (f.name, open(f, "rb"), "text/plain"))
    )

r = requests.post(
    f"{BASE_URL}/upload",
    headers=headers,
    files=multipart
)

upload_result = r.json()

print("Status code:", r.status_code)
print("\nJSON response:")
print(json.dumps(upload_result, indent=4))

Files to upload:
- src/blink_pkg.vhd
- src/modulo_counter.vhd
- src/multi_blinker_top.vhd
- src/sevenseg_decoder.vhd
- src/tick_generator.vhd
- tb/tb_multi_blinker_top.vhd
- constraints/top.xdc
Status code: 200

JSON response:
{
    "files": [
        {
            "file_id": "5b42fae8-f8d7-4f45-995c-d594c49f6dea",
            "filename": "blink_pkg.vhd",
            "size_bytes": 1102
        },
        {
            "file_id": "83e76dec-7c7e-4bf8-83ea-183ca263aeda",
            "filename": "modulo_counter.vhd",
            "size_bytes": 2333
        },
        {
            "file_id": "23c800fe-d063-46a8-978d-4cd40514fa84",
            "filename": "multi_blinker_top.vhd",
            "size_bytes": 2545
        },
        {
            "file_id": "ace32400-bafb-42ed-a02e-5e3d42f500f0",
            "filename": "sevenseg_decoder.vhd",
            "size_bytes": 1997
        },
        {
            "file_id": "d80225d1-b3a1-416c-b270-7b826be39df3",
            "filename": "tick_generator

## Extract uploaded file IDs

In [ ]:
file_map = {}

for item in upload_result["files"]:
    file_map[item["filename"]] = item["file_id"]

file_map

{'blink_pkg.vhd': '5b42fae8-f8d7-4f45-995c-d594c49f6dea',
 'modulo_counter.vhd': '83e76dec-7c7e-4bf8-83ea-183ca263aeda',
 'multi_blinker_top.vhd': '23c800fe-d063-46a8-978d-4cd40514fa84',
 'sevenseg_decoder.vhd': 'ace32400-bafb-42ed-a02e-5e3d42f500f0',
 'tick_generator.vhd': 'd80225d1-b3a1-416c-b270-7b826be39df3',
 'tb_multi_blinker_top.vhd': '63dfd489-8770-4ecf-8e04-d6b1da21759e',
 'top.xdc': '49ef9300-52ce-488f-8ab8-1dd7d6c0d414'}

## Create synthesis job

In [9]:
payload = {
    "board_id": "basys3",
    "top_entity": "multi_blinker_top",
    "file_ids": [
        file_map["blink_pkg.vhd"],
        file_map["tick_generator.vhd"],
        file_map["modulo_counter.vhd"],
        file_map["sevenseg_decoder.vhd"],
        file_map["multi_blinker_top.vhd"],
        file_map["top.xdc"]
    ]
}

r = requests.post(
    f"{BASE_URL}/jobs",
    headers={
        **headers,
        "Content-Type": "application/json"
    },
    json=payload
)

job_info = r.json()

print("Status code:", r.status_code)
print("\nJSON response:")
print(json.dumps(job_info, indent=4))

Status code: 201

JSON response:
{
    "job_id": "5611da1b-705a-4ac4-a244-7288a58bedc6",
    "status": "pending",
    "board_id": "basys3",
    "top_entity": "multi_blinker_top",
    "created_at": "2026-05-06T09:48:24.061745Z",
    "updated_at": "2026-05-06T09:48:24.061745Z",
    "error": null
}


## Save JOB_ID

In [10]:
JOB_ID = job_info["job_id"]

print(JOB_ID)

5611da1b-705a-4ac4-a244-7288a58bedc6


## Launch synthesis

In [11]:
r = requests.post(
    f"{BASE_URL}/jobs/{JOB_ID}/synthesize",
    headers=headers
)

print("Status code:", r.status_code)
print("\nJSON response:")
print(json.dumps(upload_result, indent=4))

Status code: 200

JSON response:
{
    "files": [
        {
            "file_id": "5b42fae8-f8d7-4f45-995c-d594c49f6dea",
            "filename": "blink_pkg.vhd",
            "size_bytes": 1102
        },
        {
            "file_id": "83e76dec-7c7e-4bf8-83ea-183ca263aeda",
            "filename": "modulo_counter.vhd",
            "size_bytes": 2333
        },
        {
            "file_id": "23c800fe-d063-46a8-978d-4cd40514fa84",
            "filename": "multi_blinker_top.vhd",
            "size_bytes": 2545
        },
        {
            "file_id": "ace32400-bafb-42ed-a02e-5e3d42f500f0",
            "filename": "sevenseg_decoder.vhd",
            "size_bytes": 1997
        },
        {
            "file_id": "d80225d1-b3a1-416c-b270-7b826be39df3",
            "filename": "tick_generator.vhd",
            "size_bytes": 2383
        },
        {
            "file_id": "63dfd489-8770-4ecf-8e04-d6b1da21759e",
            "filename": "tb_multi_blinker_top.vhd",
            "size_by

## Poll synthesis status

In [12]:
import time

while True:
    r = requests.get(
        f"{BASE_URL}/jobs/{JOB_ID}",
        headers=headers
    )

    data = r.json()

    print(data)

    status = data.get("status", "")

    if status in ["synthesized", "failed", "error"]:
        break

    time.sleep(2)


{'job_id': '5611da1b-705a-4ac4-a244-7288a58bedc6', 'status': 'failed', 'board_id': 'basys3', 'top_entity': 'multi_blinker_top', 'created_at': '2026-05-06T09:48:24.061745Z', 'updated_at': '2026-05-06T09:49:21.814853Z', 'error': 'Vivado exited with code 1'}


## Read reports

In [13]:
r = requests.get(
    f"{BASE_URL}/jobs/{JOB_ID}/reports",
    headers=headers
)

print(r.status_code)

reports = r.json()

reports


409


{'detail': "Reports not ready: job is in state 'failed'"}

## Launch simulation

In [ ]:
sim_payload = {
    "testbench_file_id": file_map["tb_multi_blinker_top.vhd"],
    "testbench_top": "tb_multi_blinker_top",
    "runtime": "1us"
}

r = requests.post(
    f"{BASE_URL}/jobs/{JOB_ID}/simulate",
    headers={
        **headers,
        "Content-Type": "application/json"
    },
    json=sim_payload
)

print(r.status_code)

sim_result = r.json()

sim_result


## Read simulation output

In [ ]:
r = requests.get(
    f"{BASE_URL}/jobs/{JOB_ID}",
    headers=headers
)

print(r.status_code)

r.json()
